# **Map of ISS**

In [27]:
from google.colab import drive
drive.mount('/content/drive')

import folium
from datetime import datetime
from skyfield.api import load
from spacetrack import get_iss, get_pointing_coords

#known coordinates
irbene_coords = [57.5535, 21.8545]
tx_coords = [52.23167, 21.00639]

start_date = "2025-03-04 00:04:31"
end_date = "2025-03-05 00:04:31"
full_date = datetime.strptime(end_date, '%Y-%m-%d %H:%M:%S')

iss, l1, l2 = get_iss(start_date, end_date,full_date)

#time of observation
ts = load.timescale()
start = datetime(2025, 3, 5, 0, 4, 31)

times = [ts.utc(start.year, start.month, start.day, start.hour, start.minute, start.second + s)
           for s in range(0, 630, 30)]

traj = []
for t in times:
    subpoint = iss.at(t).subpoint()
    traj.append([subpoint.latitude.degrees, subpoint.longitude.degrees])

map = folium.Map(location=[50.0, 15.0], zoom_start=5)
folium.TileLayer(
    tiles='http://mt0.google.com/vt/lyrs=m&hl=en&x={x}&y={y}&z={z}',
    attr='Google Maps'
).add_to(map)

# Draw line and markers
folium.PolyLine(traj, color="red", weight=1).add_to(map)

for i, coord in enumerate(traj):
    marker = folium.CircleMarker(
        location=coord,
        radius=2,
        color="red",
        fill=True,
        fill_color="red",
        fill_opacity=1,
    )

    if i % 5 == 0:
        time_str = times[i].utc_strftime('%H:%M:%S')
        dt = times[i].utc_datetime()

        alt, az = get_pointing_coords(iss, "LV614HBA", dt)

        txt = f"{time_str}<br>el: {alt:.2f}°"
        fontsize = "font-size: 14px"
        folium.Tooltip(txt, style=fontsize, permanent=True, direction='bottom').add_to(marker)

    marker.add_to(map)

folium.Marker(
    location=irbene_coords,
    icon=folium.Icon(color='darkblue', icon='satellite-dish', prefix='fa'),
).add_to(map)

folium.Marker(
    location=tx_coords,
    icon=folium.Icon(color='orange', icon='broadcast-tower', prefix='fa'),
).add_to(map)

key = '''
<div style="position: fixed; bottom: 50px; left: 50px; width: 160px; height: 90px;
            background-color: rgba(255, 255, 255, 0.5); z-index:9999; border:1px solid black; padding: 10px;">
    <b>Key</b><br>
    <i class="fa fa-satellite-dish" style="color:darkblue"></i> LOFAR (VIRAC)<br>
    <i class="fa fa-broadcast-tower" style="color:orange"></i> DAB Transmitter
</div>
'''
map.get_root().html.add_child(folium.Element(key))

folium.PolyLine([tx_coords, irbene_coords], weight=2, dash_array='5, 5', opacity=0.7).add_to(map)

map

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data downloaded, converting to Skyfield objects.


In [1]:
#!pip install skyfield lofarantpos
import sys
sys.path.insert(0, '/content/drive/MyDrive/Colab THESIS')

import importlib
import spacetrack
importlib.reload(spacetrack)

<module 'spacetrack' from '/content/drive/MyDrive/Colab THESIS/spacetrack.py'>